In [4]:
! pip install -r requirements.txt

In [5]:
from langchain_community.document_loaders import PyPDFLoader

In [6]:
# Load the PDF Doc

loader = PyPDFLoader('/home/n1mun0va/luxdev/FinanceBill2025-RAG-Application/The_Finance_Bill_2025.pdf')
docs = loader.load()

Ignoring wrong pointing object 511 0 (offset 0)
Ignoring wrong pointing object 4016 0 (offset 0)


In [7]:
print(docs[0])

page_content='SPECIAL ISSUE 
NATION I. '01.INCI I.FOR 
k P OR TI NC 
LIPRAPY  
 
   
Kenya Gazette Supplement No. 63 (National Assembly Bills No. 19) 
REPUBLIC OF KENYA 
KENYA GAZETTE SUPPLEMENT 
NATIONAL ASSEMBLY BILLS, 2025 
NAIROBI, 6th May, 2025 
CONTENT 
Bill for Introduction into the National Assembly— 
PAGE 
The Finance Bill, 2025  	 335 
NATIONAL COUNCIL FOR 
LAW REPORTING 
0 9 MAY 2025 
LIBRARY ARCHIVE 
PRINTED AND PUBLISHED BY THE GOVERNMENT PRINTER, NAIROBI' metadata={'producer': 'OmniPageCSDK18', 'creator': 'HP Smart Document Scan Software 3.70', 'creationdate': '2025-05-09T13:05:43+03:00', 'moddate': '2025-05-09T14:00:52+03:00', 'source': '/home/n1mun0va/luxdev/FinanceBill2025-RAG-Application/The_Finance_Bill_2025.pdf', 'total_pages': 135, 'page': 0, 'page_label': '1'}


In [8]:
# Generates chunks of the document

from langchain_text_splitters import RecursiveCharacterTextSplitter

textsplitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap = 200)

chunks = textsplitter.split_documents(docs)
print(len(chunks))

367


In [9]:
# Generates vector embeddings for each chunk

from langchain_ollama import OllamaEmbeddings

embeddings =OllamaEmbeddings(model='nomic-embed-text')

# Stores the chunks in a chroma vector db 
from langchain_chroma import Chroma

db =Chroma.from_documents(documents = chunks, embedding=embeddings, persist_directory='./chroma_db')

In [10]:
# Create a retriever to find relevant chunks based on a question
retriever = db.as_retriever()

In [11]:
# Intiating my LLM

from langchain_ollama.chat_models import ChatOllama

llm = ChatOllama(model= 'mistral:7b')

In [12]:
from langchain.prompts import ChatPromptTemplate, PromptTemplate

QUERY_PROMPT = PromptTemplate(
    input_variables=['question'],
    template='''You are an AI language model assistant. Your task is to generate five
    different versions of the given user question to retrieve relevant documents from
    a vector database. By generating multiple perspectives on the user question, your
    goal is to help the user overcome some of the limitations of the distance-based
    similarity search. Provide these alternative questions separated by newlines.
    Original question: {question}''',
)

In [13]:
# # Creating a retriever to find relevant chunks based on a question
from langchain.retrievers.multi_query import MultiQueryRetriever

retriever = MultiQueryRetriever.from_llm(db.as_retriever(),
                                         llm,
                                         prompt = QUERY_PROMPT)

#RAG Prompt
template = '''Answer the question based ONLY on the following context:
{context}
Question= {question}
'''

prompt = ChatPromptTemplate.from_template(template)

In [14]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = (
    {'context': retriever, 'question':RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)

In [15]:
chain.invoke('What does this document talk about?',config={'verbose':True})

" The document discusses the requirements for a master file under subsection (3), as outlined in The Finance Bill, 2025. The master file is expected to contain various details about a group, such as an overview of the group, its growth engines, supply chain information, research and development policy, each constituent entity's contribution to value creation, information on intangible assets and associated intercompany agreements, transfer of intangible assets within the group during the tax period, financing activities, consolidated financial statements, and any tax rulings made in respect of the group. The document does not specify what the Finance Bill is or who the Commissioner is."

In [ ]:

chain.invoke('What is the main idea of this document',config={'verbose':True})